In [7]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 103.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 154.2 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [8]:
!pip install langchain-google-genai

In [9]:
import os
from dotenv import load_dotenv

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_544/2793349658.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader

In [19]:
load_dotenv()

True

In [13]:
def get_embeddings():
    return GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001"
    )

In [15]:
def build_or_load_index():
    """Create a FAISS index from the PDF, or load it if already created."""

    embeddings = get_embeddings()
    index_file = os.path.join(INDEX_DIR, "index.faiss")

    if os.path.exists(INDEX_DIR) and os.path.exists(index_file):
        print("Loading existing PDF knowledge base...")
        vector_store = FAISS.load_local(
            INDEX_DIR,
            embeddings,
            allow_dangerous_deserialization=True
        )
    else:
        print("Reading PDF and building knowledge base...")

        loader = PyPDFLoader(PDF_FILE)
        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=100
        )
        chunks = splitter.split_documents(documents)

        vector_store = FAISS.from_documents(chunks, embeddings)
        vector_store.save_local(INDEX_DIR)

    return vector_store.as_retriever(search_kwargs={"k": 3})

In [16]:
def create_rag_chain(retriever):
    prompt = ChatPromptTemplate.from_template(
        """
        You are a helpful assistant for ABC Company employees.

        Answer the user's question using ONLY the context below.

        If the answer is not present in the context, reply:
        "I don't know based on the provided ABC Company document."

        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    )

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash-lite",
        temperature=0
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    return (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

In [20]:
def main():
    print("ABC Company PDF RAG Chatbot")
    print("Type 'exit' to quit.\n")

    retriever = build_or_load_index()
    rag_chain = create_rag_chain(retriever)

    while True:
        question = input("You: ").strip()

        if question.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        if not question:
            continue

        answer = rag_chain.invoke(question)
        print("Chatbot:", answer, "\n")

In [22]:
PDF_FILE = "/content/ABC_Company_Profile.pdf"
INDEX_DIR = "faiss_index"

if __name__ == "__main__":
    main()

ABC Company PDF RAG Chatbot
Type 'exit' to quit.

Loading existing PDF knowledge base...
You: Headquarters?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Chatbot: 245 Market Street, Suite 800, Austin, TX 78701 

You: What's your achievements in the comapny?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Chatbot: I don't know based on the provided ABC Company document. 

You: Milestones?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Chatbot: Based on the provided ABC Company document, the milestones are:

* **2012** – ABC founded by Daniel Brooks in Austin, Texas
* **2015** – Launched ABC Cloud and reached 100 customers
* **2018** – Opened European office in Dublin, Ireland
* **2021** – Released ABC Insight analytics platform
* **2024** – Surpassed 800 clients worldwide
* **2025** – Named a 'Top 50 Workplace' in Texas 

You: How can i contact you?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Chatbot: You can contact ABC Inc. using the following contact information provided in the document:

* **Address:** 245 Market Street, Suite 800, Austin, TX 78701, USA
* **Phone:** +1 (512) 555-0142
* **Email:** info@abc-example.com 

You: exit
Chatbot: Goodbye!
